# Analysis pipeline
Script originally created using [Honeybee](https://honeybee-lang.org) (version 0.5.0).
Please cite:
- Justin Lubin, Parker Ziegler, and Sarah E. Chasins. 2025. Programming by Navigation. Proc. ACM Program. Lang. 9, PLDI, Article 165 (June 2025), 28 pages. https://doi.org/10.1145/3729264

## Parameters

Before running your code, please set the following parameters!

In [1]:
# PARAMETER: The folder containing the Bismark reference genome to align against (default: "/Users/barb/Documents/genomes/bismark_genome_folder")
BISMARK_GENOME_FOLDER = "/mnt/Shared1/genomes/EMseq_hg38"

# PARAMETER: The forward adapter, by default the Illumina universal (default: "AGATCGGAAGAGCACACGTCTGAACTCCAGTCA")
FORWARD_ADAPTER = "AGATCGGAAGAGCACACGTCTGAACTCCAGTCA"

# PARAMETER: The reverse adapter, by default the Illumina universal (default: "AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT")
REVERSE_ADAPTER = "AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT"

# PARAMETER: The number of cores that you want FastQC to use (default: 4)
FASTQC_CORES = 4

# PARAMETER: The number of cores that you want Bismark to use (default: 4)
BISMARK_CORES = 60

# PARAMETER: The suffix at the end of the filenames for the forward reads (default: "_R1")
FORWARD_READ_SUFFIX = "_R1"

# PARAMETER: The suffix at the end of the filenames for the reverse reads (default: "_R2")
REVERSE_READ_SUFFIX = "_R2"

## Initialization code



In [2]:
from dataclasses import dataclass
import datetime
import glob
import os
import polars as pl
import re

def stem(path):
    while True:
        path, ext = os.path.splitext(os.path.basename(path))
        if not ext:
            return path

def carry_over(src_object, dst_object, *, file=None):
    os.makedirs(dst_object.path, exist_ok=True)
    
    def carry_one(f):
        src = f"{src_object.path.rstrip('/')}/{f}"
        if not src_object.path.startswith("/"):
            src = f"../../{src}"  # relative to dst
        dst = f"{dst_object.path}/{f}"
        if os.path.islink(src):
            src = os.readlink(src)
        os.symlink(src=src, dst=dst)

    if file is None:
        for file in os.listdir(src_object.path):
            carry_one(file)
    else:
        carry_one(file)

def save(src, dst):
    dir = os.path.dirname(dst)
    os.makedirs(dir, exist_ok=True)
    os.symlink(src=os.path.abspath(src), dst=dst)

@dataclass
class LocalEmSeq:
    "EM-seq"

    path: str
    """Path to the directory containing the raw EM-seq data

    @example:/Users/barb/Desktop/MyExperiment/raw-fastq-reads/

    This directory should contain files ending with `.fastq` or `.fastq.gz`."""

@dataclass
class EmSeqNoRef:
    "@intermediate:EM-seq data (without converted reference)"

    path: str
    """Data path

    Directory structure:
    - *.fastq: raw EM-seq data
    """

@dataclass
class SeqReads:
    """@intermediate:Sequencing reads

    The goal of this step is to process sequencing "reads."

    A "read" is either a short (a few hundred base pairs) or long (a few
    thousand base pairs) snippet of DNA produced by a sequencer. This DNA can
    be genomic DNA (perhaps that has undergone some processing, such as in
    [ATAC-seq](https://www.nature.com/articles/nmeth.2688)), or it can be
    derived from RNA in a cell (as in RNA-sequencing).

    Reads are stored typically stored in the `.fastq` (uncompressed) or
    `.fastq.gz` (compressed) file format. Before being able to get a table of
    information (e.g. about chromatin accessibility, methylation status, or
    transcription levels), reads first need to be _preprocessed_. Most
    preprocessing is method-specific, but some methods share commonalities, such
    as _quality control_ (QC) checks."""

    path: str
    """Data path

    Directory structure:
    - *.fastq: raw data
    - reference/*: files for the reference for the reads"""

    qc: bool
    trimmed: bool
    long: bool
    type: str

@dataclass
class ChunkReads:
    """@intermediate:Split sequencing reads

    The Goal of this step is to split the fastq files into files of size "M" or
    **Work in progress "N" number of files **

    EM-seq files are large leading to limitations in effectively running parallel
    instances of Bismark. Each increase in parallelization leads to a linear decrease
    in compute time. Ie. --parallel 4 decreases compute time to ~25-30% 
    (https://felixkrueger.github.io/Bismark/options/alignment/)

    Smaller file sizes will allow for increased parallelization, speeding up the
    compute time.
    
    """
    path: str
    chunk: bool
    # size_chunk: bool
    # num_chunk: bool
    fil_size: int
    # fil_num: int



@dataclass
class SeqAlignment:
    """Sequence alignment

    [SAM files](https://doi.org/10.1093/bioinformatics/btp352) are
    **uncompressed** files that describe an alignment of reads to a reference
    genome. These alignments may or may not be "sorted" by nucleotide position
    and may or may not be "indexed"; an "index" is a supplementary file to the
    SAM file that allows for the computer to quickly access various information
    from the alignment for viewing in, for example, the
    [Integrative Genomics Viewer (IGV)](https://igv.org/).

    The equivalent **compressed** files are BAM files."""

    path: str

    compressed: bool
    type: str

@dataclass
class MethylationDedup:
    """Methylation Deduplication

    The goal of this step is to remove PCR duplicates from the aligned reads which 
    can arise from excess PCR amplification. Deduplication can help reduce technical
    bias in the library prep and sequencing step. AUTO-DETECT is set by default.

    The percent methylation can be skewed without this step. 
    """

    path: str
    dedup: bool


@dataclass
class MethylationCalls:
    """Methylation calls (per-cytosine methylation)

    The goal of this step is to produce a table of "methylation calls"; that is,
    a table where each row corresponds to a cytosine in the provided reference
    genome, and each column of the table provides information about how often
    that cytosine was methylated in the reads being analyzed.

    This information is useful for downstream plotting or simply to look at the
    table itself to understand patterns of methylation!"""

    path: str

## EM-seq



In [3]:
LOCAL_EM_SEQ = LocalEmSeq(
    path="output/trimmed",
)

## Load raw EM-seq data already present on your computer

The raw EM-seq files are typically in the `.fastq` or `.fastq.gz` file
format.

In [4]:
EM_SEQ_NO_REF = EmSeqNoRef(
    path="output/010-load_local_em_seq",
)

carry_over(LOCAL_EM_SEQ, EM_SEQ_NO_REF)

FileExistsError: [Errno 17] File exists: '../../output/trimmed/PC_EM_A2_S9_R2.fastq.gz' -> 'output/010-load_local_em_seq/PC_EM_A2_S9_R2.fastq.gz'

## Use an existing (in silico) [EM-converted Bismark reference genome](https://felixkrueger.github.io/Bismark/bismark/genome_preparation/)

In order to perform alignment of EM-seq reads, we need a version of
the reference genome that has undergone _in silico EM conversion_; or, in
other words, that has all Cs converted to Ts (and Gs to As in the reverse
reads) in the `.fasta` files (computationally).

If you already have a reference genome that has undergone in silico EM
conversion for Bismark, you don't need to redo that step! After downloading
the completed script, simply put in the path to where that reference genome
is stored in the parameter below.

In [5]:
SEQ_READS4 = SeqReads(
    path="output/020-use_existing_bismark_reference",
    qc=False,
    trimmed=False,
    long=False,
    type="em",
)

carry_over(EM_SEQ_NO_REF, SEQ_READS4)

save(
    BISMARK_GENOME_FOLDER,
    f"{SEQ_READS4.path}/reference",
)

FileExistsError: [Errno 17] File exists: '../../output/010-load_local_em_seq/PC_EM_A2_S9_R2.fastq.gz' -> 'output/020-use_existing_bismark_reference/PC_EM_A2_S9_R2.fastq.gz'

## Run quality control checks on RNA-seq data with [FastQC](https://www.bioinformatics.babraham.ac.uk/projects/fastqc/)

FastQC produces two HTML reports for each sample: one for the forward reads
and one for the reverse reads. These HTML reports can be individually opened
and inspected in your web browser.

The Harvard Chan Bioinformatics Core provides a
[useful tutorial](https://hbctraining.github.io/Intro-to-rnaseq-hpc-salmon/lessons/qc_fastqc_assessment.html#assessing-quality-metrics)
for assessing the outputs of FastQC.

### Citation

- Simon Andrews. FastQC: a quality control tool for high throughput sequence data. (2010). Available online at: http://www.bioinformatics.babraham.ac.uk/projects/fastqc

In [6]:
SEQ_READS3 = SeqReads(
    path="output/030-fastqc",
    qc=True,
    trimmed=False,
    long=False,
    type="em",
)

carry_over(SEQ_READS4, SEQ_READS3)

fastqs = " ".join(glob.glob(f"{SEQ_READS4.path}/*.fastq*"))

# !fastqc -t {FASTQC_CORES} -o {SEQ_READS3.path} {fastqs}

FileExistsError: [Errno 17] File exists: '../../output/020-use_existing_bismark_reference/PC_EM_A2_S9_R2.fastq.gz' -> 'output/030-fastqc/PC_EM_A2_S9_R2.fastq.gz'

## Remove sequencing adapters using [cutadapt](https://cutadapt.readthedocs.io/en/stable/).

[Adapter trimming](https://knowledge.illumina.com/library-preparation/general/library-preparation-general-reference_material-list/000001314)
removes adapter sequences that are present due to a read length being
longer than the insert size of the sequence in a sequencer.

### Citation

- Marcel Martin. Cutadapt removes adapter sequences from high-throughput sequencing reads. EMBnet.Journal, 17(1):10-12, May 2011. http://dx.doi.org/10.14806/ej.17.1.200

In [7]:
SEQ_READS2 = SeqReads(
    path="output/040-cutadapt",
    qc=False,
    trimmed=True,
    long=False,
    type="em",
)

carry_over(SEQ_READS3, SEQ_READS2)

FileExistsError: [Errno 17] File exists: '../../output/030-fastqc/PC_EM_A2_S9_R2.fastq.gz' -> 'output/040-cutadapt/PC_EM_A2_S9_R2.fastq.gz'

## Run quality control checks on RNA-seq data with [FastQC](https://www.bioinformatics.babraham.ac.uk/projects/fastqc/)

FastQC produces two HTML reports for each sample: one for the forward reads
and one for the reverse reads. These HTML reports can be individually opened
and inspected in your web browser.

The Harvard Chan Bioinformatics Core provides a
[useful tutorial](https://hbctraining.github.io/Intro-to-rnaseq-hpc-salmon/lessons/qc_fastqc_assessment.html#assessing-quality-metrics)
for assessing the outputs of FastQC.

### Citation

- Simon Andrews. FastQC: a quality control tool for high throughput sequence data. (2010). Available online at: http://www.bioinformatics.babraham.ac.uk/projects/fastqc

In [8]:
SEQ_READS = SeqReads(
    path="output/050-fastq",
    qc=True,
    trimmed=True,
    long=False,
    type="em",
)

carry_over(SEQ_READS2, SEQ_READS)

fastqs = " ".join(glob.glob(f"{SEQ_READS2.path}/*.fastq*"))

!fastqc -t {FASTQC_CORES} -o {SEQ_READS.path} {fastqs}

FileExistsError: [Errno 17] File exists: '../../output/040-cutadapt/PC_EM_A2_S9_R2.fastq.gz' -> 'output/050-fastq/PC_EM_A2_S9_R2.fastq.gz'

In [9]:
CHUNK_READS = ChunkReads(
    path = "output/055-fastq-chunk",
    chunk = True,
    fil_size = 200_000_000,
)

!mkdir -p {CHUNK_READS.path}

for path in glob.glob(f"{SEQ_READS.path}/*{FORWARD_READ_SUFFIX}.fastq.gz"):
    sample_name = stem(path).removesuffix(FORWARD_READ_SUFFIX)
    print(f"Running seqkit split2 on '{sample_name}'...")
    !mkdir -p {CHUNK_READS.path}/{sample_name}
    !seqkit split2 \
        --threads 40 \
        -1 {SEQ_READS.path}/{sample_name}{FORWARD_READ_SUFFIX}.fastq.gz \
        -2 {SEQ_READS.path}/{sample_name}{REVERSE_READ_SUFFIX}.fastq.gz \
        -s {CHUNK_READS.fil_size} \
        -O {CHUNK_READS.path}/{sample_name}


## Align EM-converted reads with [Bismark](https://felixkrueger.github.io/Bismark/bismark/alignment/)

Bismark uses [bowtie2](https://bowtie-bio.sourceforge.net/bowtie2/index.shtml)
under the hood to align wet-lab EM-converted sample reads to an in silico
EM-converted reference.

### Citations

- Krueger F, Andrews SR. Bismark: a flexible aligner and methylation caller for Bisulfite-Seq applications. Bioinformatics. 2011 Jun 1;27(11):1571-2. doi: 10.1093/bioinformatics/btr167. Epub 2011 Apr 14. PMID: 21493656; PMCID: PMC3102221.
- Langmead B, Salzberg SL. Fast gapped-read alignment with Bowtie 2. Nat Methods. 2012 Mar 4;9(4):357-9. doi: 10.1038/nmeth.1923. PMID: 22388286; PMCID: PMC3322381.

In [11]:
##Old script
# SEQ_ALIGNMENT = SeqAlignment(
#     path="output/060-bismark",
#     compressed=True,
#     type="em",
# )

# for path in glob.glob(f"{SEQ_READS.path}/*{FORWARD_READ_SUFFIX}.fastq.gz"):
#     sample_name = stem(path).removesuffix(FORWARD_READ_SUFFIX)
    
#     !mkdir -p {SEQ_ALIGNMENT.path}/temp
#     !rm -rf {SEQ_ALIGNMENT.path}/temp
#     !mkdir -p {SEQ_ALIGNMENT.path}/temp

#     already_ran = False
#     for filename in os.listdir(SEQ_ALIGNMENT.path):
#         if sample_name in filename:
#             already_ran = True
#     if already_ran:
#         continue

#     print(f"Running bismark on '{sample_name}'...")
#     !bismark \
#         --bam \
#         --parallel {1} \
#         --quiet \
#         --temp_dir {SEQ_ALIGNMENT.path}/temp \
#         --genome {SEQ_READS.path}/reference \
#         -o {SEQ_ALIGNMENT.path}/temp \
#         -1 {SEQ_READS.path}/{sample_name}{FORWARD_READ_SUFFIX}.fastq.gz \
#         -2 {SEQ_READS.path}/{sample_name}{REVERSE_READ_SUFFIX}.fastq.gz

#     !mv {SEQ_ALIGNMENT.path}/temp/* {SEQ_ALIGNMENT.path}
    

In [ ]:
#reoptimized for atomic writing, caching, and chunking
SEQ_ALIGNMENT = SeqAlignment(
    path="output/060-bismark",
    compressed=True,
    type="em",
)

!mkdir -p {SEQ_ALIGNMENT.path}
for path in sorted(glob.glob(f"{SEQ_READS.path}/*{FORWARD_READ_SUFFIX}.fastq.gz")): #this script if for chunked fastqs
    path_basename = stem(path).removesuffix(FORWARD_READ_SUFFIX)
    input_path = f"{CHUNK_READS.path}/{path_basename}"
    
    !mkdir -p {SEQ_ALIGNMENT.path}/{path_basename}
    for filename in sorted(glob.glob(f"{input_path}/*{FORWARD_READ_SUFFIX}*.fastq.gz")):
        !mkdir -p {SEQ_ALIGNMENT.path}/{path_basename}/temp
        !rm -rf {SEQ_ALIGNMENT.path}/{path_basename}/temp
        !mkdir -p {SEQ_ALIGNMENT.path}/{path_basename}/temp
        
        #caching step
        part_match = re.search(r'(part_\d+)', filename)
        part_num = part_match.group(1) if part_match else "unknown_part"

        already_ran = False
        if os.path.exists(f"{SEQ_ALIGNMENT.path}/{path_basename}"):
            for output_file in os.listdir(f"{SEQ_ALIGNMENT.path}/{path_basename}"):
                if part_num in output_file and output_file.endswith(".bam"):
                    already_ran = True
                    break
        
        if already_ran:
            print(f"{filename} has already been run")
            continue

        chunk_R1=filename
        chunk_R2=filename.replace("_R1.part_", "_R2.part_", 1)

        run_name = chunk_R1.split("/")[2] +"_"+chunk_R1.split(".")[1]
        print(f"Running bismark align of {run_name}...")
        #note: for 200gb file size, parallel 15 ~140 gb ram, 64 threads; parallel 12 ~140gb ram, 57 threads. 13 is optimal? 
        !bismark \
            --bam \
            --parallel {12} \
            --quiet \
            --temp_dir {SEQ_ALIGNMENT.path}/{path_basename}/temp \
            --genome {SEQ_READS.path}/reference \
            -o {SEQ_ALIGNMENT.path}/{path_basename}/temp \
            -1 {chunk_R1} \
            -2 {chunk_R2}

        for filename in os.listdir(f"{SEQ_ALIGNMENT.path}/{path_basename}/temp"):
            if ".temp" in filename:
                continue
            !mv "{SEQ_ALIGNMENT.path}/{path_basename}/temp/{filename}" "{SEQ_ALIGNMENT.path}/{path_basename}"


## Call methylation with the [Bismark Methylation Extractor](https://felixkrueger.github.io/Bismark/bismark/methylation_extraction/)

This step also produces bedGraph files and "coverage" files using
[bismark2bedGraph](https://felixkrueger.github.io/Bismark/bismark/methylation_extraction/#optional-bedgraph-output).

The most important output files are the **coverage** files, as they contain
the number of methylated and unmethylated reads at each CpG. These files
enable essentially any downstream analysis of interest.

In [ ]:
#only optimized for multiple. Need to change params for singular.
DEDUPLICATION = MethylationDedup(
    path="output/070-bismark_deduplication",
    dedup= True,
)

!mkdir -p {DEDUPLICATION.path}
for path in sorted(glob.glob(f"{SEQ_ALIGNMENT.path}/*")):
    path_basename = stem(path)
    multi_input = glob.glob(f"{path}/*.bam") #[f"{path}/{i}" for i in glob.glob(f"{path}/*.bam")]

    multi_input_string = ""
    for i in multi_input:
        multi_input_string += f" {i}"

    print(f"Running deduplicate_bismark on '{path_basename}'...")
    !deduplicate_bismark \
        --bam \
        --multiple \
        --outfile {path_basename} \
        --output_dir {DEDUPLICATION.path} \
        {multi_input_string}

In [ ]:
# GOAL = MethylationCalls(
#     path="output/070-bismark_methylation_extractor",
# )

# SEQ_ALIGNMENT: SeqAlignment,
# GOAL: MethylationCalls,

# for path in glob.glob(f"{SEQ_ALIGNMENT.path}/*.bam"):
#     print(f"Running bismark_methylation_extractor on '{path}'...")
#     !bismark_methylation_extractor \
#         --parallel {max(BISMARK_CORES // 3, 1)} \
#         --gzip \
#         --bedGraph \
#         -o {GOAL.path} \
#         {path}